# Summarize Conservation by Basin

This notebook provides a step-by-step guide on utilizing the OpenET API's */geodatabase/metadata/ids* endpoint to retrieve field IDs contained within a specified region. Then make an additional query to the */geodatabase/timeseries* endpoint to get monthly evapotranspiration and precipitation data for all fields within

## To complete this notebook you will need:
1. [OpenET Account](https://account.etdata.org/login)
2. [OpenET API Key](https://auth.etdata.org/dashboard)
3. A valid GeoJSON file. This demo uses the HUC-12 watershed boundary of the Little Shasta River.


To apply this same workflow for another region:
* Convert a shapefile to GeoJSON using **[MapShaper](https://mapshaper.org/)**
* Evaluate GeoJSON compatibility with OpenET data servicesusing **[GeoJSONLint](https://geojsonlint.com/)**


# Collab Notebook Outline

1.   **Import Python dependencies**
2. **Configure API Key**: Copy your API Key and paste in prompt to access OpenET data.
3.   **Load boundary as GeoJSON**: Load a GeoJson and create temporary asset ID to pass Geometry to API.
4.  **Make /geodatabase/metadata/ids API request** to retrieve the field ids associated with all fields within the watershed..
5.  **Make /geodatabase/timeseries API request** to save time series of evapotranspiration and precipitation for all fields within the watershed.: Construct and send a request to retrieve ET, and Pr data for all field boundaries within the defined region of interest.
6.  **Make additional API requests** to save field boundaries
7. **Combine data to analyze which fields were impacted by curtailments.**




##### 1. Import Python dependencies


In [ ]:
import calendar
import getpass
import gzip
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import requests
import seaborn as sns
import shapely
# import geemap


##### 2. Configure your API Key to access OpenET data

Go to your [OpenET API account](https://account.etdata.org/settings/api) page and copy the API Key. { run: "auto" }
Then paste your API Key below after running this cell

In [ ]:


YOUR_API_KEY = getpass.getpass("Paste your API key here --> ")


header = {"Authorization": YOUR_API_KEY}

#### 3. Load boundary as GeoJSON
In the files sidetab, you can right-click your file and select the option 'Copy path', which you can paste into the text field.


In [ ]:

geojson_path = "geospatial_data/workshop_llsr.geojson" # @param {"type": "string", "placeholder": "Insert path here"}
# Get shape from GeoJSON

# Read the GeoJSON
geojson = gpd.read_file(geojson_path)

# Extract the coordinates of the GeoJSON with shapely.get_coordinates to a list to pass to the API.
# The code flattens the matrix and puts the coordinates into a 1D list (format required for the API).
geometry = shapely.get_coordinates(geojson.geometry).flatten().tolist()

##### Retrieve Field IDs contained within GeoJSON


In [ ]:

# This provides the API with your API key.
api_header = {"AUTHORIZATION": YOUR_API_KEY}
# This provides the parameters for the request.
api_query = {
    "geometry": geometry
}

# This is the request itself.
api_request = requests.post(
    url="https://openet-api.org/geodatabase/metadata/ids",
    headers=api_header,
    json=api_query
)

# Parse the response since it is compressed.
fields = eval(gzip.decompress(api_request.content).decode())

In [ ]:
# set your API key before making the request
header = {"Authorization": YOUR_API_KEY}

# endpoint arguments
args = {
  "field_ids":fields
}

# query the api
resp = requests.post(
    headers=header,
    json=args,
    url="https://openet-api.org/geodatabase/metadata/properties"
)

# unzip the data
meta_data = eval(gzip.decompress(resp.content).decode())

##### Retrieve monthly timeseries data for 2017-2022


In [ ]:

api_query = {
    "field_ids": fields,
    "date_range": ["2016-01-01", "2025-12-31"],
    "interval": "monthly",
    "variables": ["et", "pr"],
    "models": ["ensemble"],
    "file_format": "JSON"
}

api_request = requests.post(
    url="https://openet-api.org/geodatabase/timeseries",
    headers=api_header,
    json=api_query
)

timeseries = eval(gzip.decompress(api_request.content).decode())

##### Look at data time series data


In [ ]:
# 'value_mm' units are in mm
timeseries[0]

##### Read the timeseries data as a Pandas dataframe


In [ ]:
#read dataframe from json
df_meta =pd.DataFrame.from_records(meta_data)[["field_id","hectares"]]
df_meta["acres"] = df_meta["hectares"]*0.404686
df = pd.DataFrame.from_records(timeseries)
df["time"] = pd.to_datetime(df["time"])

# separate et and precipitation then merge dataframes
et_df = df.query("collection == 'ensemble_et'")
et_df = et_df.drop(columns="collection").rename(columns={"value_mm": "et"})

pr_df = df.query("collection == 'gridmet_pr'")
pr_df = pr_df.drop(columns="collection").rename(columns={"value_mm": "pr"})

df = pd.merge(et_df, pr_df, how="inner", on=["field_id", "time"])
cols_to_convert = ["et", "pr"]
df[cols_to_convert] = df[cols_to_convert] / 25.4


df_all = pd.merge(df, df_meta[["field_id","acres"]], how="inner", on=["field_id"])
df_all['year']=df_all.time.dt.year


### Calculate monthly ETn

While there are more advanced approaches. This method assumes:
* 100% of precipitation is during growing season
* minimal to 0 soil water carry over
* The simplifying asumptions calculate ETnet as max(0, ET-Pr)

In [ ]:
def calculate_etn(df):
    return max(0, df["et"] - df["pr"])

df_all["etn"] = df_all.apply(calculate_etn, axis=1)
df_all.query("etn > 0").head()

#### Aggregate sum of ETn each year for March-October


In [ ]:
# Aggregate sum of ETn each year for March-October

# Set a year and month columns to group by.
df_all["year"] = df_all["time"].dt.year
df_all["month"] = df_all["time"].dt.month

# Filter the desired date range.
df_m_to_o = df_all.query("month >= 3 and month <= 10").copy() # Explicitly create a copy
df_m_to_o["et_acreft"]=df_m_to_o["et"]*df_m_to_o["acres"]/12 # convert in to ft
df_m_to_o["etn_acreft"]=df_m_to_o["etn"]*df_m_to_o["acres"]/12 # convert in to ft


# Aggregate the sum of ETn for each year and field.
etn_yearly_sums = df_m_to_o.groupby(["field_id", "year"])[["etn_acreft","et_acreft"]].sum().reset_index()

#### Plot ETn by year relative to ET

In [ ]:
plt.style.use('ggplot')
fig, axs = plt.subplots(1,1,figsize=(5,5))
etn_yearly_sums[["year","etn_acreft","et_acreft"]].rename(columns={"etn_acreft":"ETn","et_acreft":"ET"}).groupby("year").sum().reset_index().set_index("year").plot(label=["ETn","ET"],ax=axs)
plt.ylabel("Mar-October Volume (acre-ft)")
plt.xlabel(" ")
plt.ylim(0,4000)
plt.title("Lower Shasta Consumptive Use")